# Q1

## Q1 code

According to lecture_3_SQL_and_SparkSQL notes for 14763 systems and Tool Chains for AI Engineers

In [1]:
!pip install wget

In [2]:
!python -m wget https://www.andrew.cmu.edu/user/mfarag/static/shopping_trends.csv


Saved under shopping_trends.csv


In [3]:
from pyspark.sql import SparkSession
import wget

spark = SparkSession.builder \
    .appName("Shopping_Trends_Stats") \
    .getOrCreate()

In [4]:
df = spark.read.csv("shopping_trends.csv", header=True, inferSchema=True)

In [5]:
df.printSchema()
df.describe().show()

root
 |-- Customer ID: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Item Purchased: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Purchase Amount (USD): integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Review Rating: double (nullable = true)
 |-- Subscription Status: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Shipping Type: string (nullable = true)
 |-- Discount Applied: string (nullable = true)
 |-- Promo Code Used: string (nullable = true)
 |-- Previous Purchases: integer (nullable = true)
 |-- Preferred Payment Method: string (nullable = true)
 |-- Frequency of Purchases: string (nullable = true)

+-------+------------------+-----------------+------+--------------+-----------+---------------------+--------+----+------+------+------------

## Q1 output

Refer to q1.png

# Q2

According to lecture_5_Neo4J notes for 14763 systems and Tool Chains for AI Engineers

According to https://neo4j.com/docs/cypher-manual/current/constraints/managing-constraints/

According to https://neo4j.com/docs/python-manual/current/#python-driver-session-run

## Q2 code

In [6]:
!pip install neo4j

In [7]:
from neo4j import GraphDatabase
import pandas as pd

df = pd.read_csv("shopping_trends.csv")
rows = df.to_dict("records")

class Neo4JConnection:
    def __init__(self, uri, user, password): 
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        if self.driver:
            self.driver.close()

    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return result.data()
conn = Neo4JConnection("bolt://localhost:7687", "neo4j", "Zeroking@2025")
# Reference: https://neo4j.com/docs/cypher-manual/current/constraints/managing-constraints/
conn.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE") # ensure each item appears only oncew in the graph
conn.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (i:Item) REQUIRE i.name IS UNIQUE")

[]

In [8]:
for idx, row in df.iterrows():# Reference:https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html
    user_properties = {
        "user_id": int(row["Customer ID"]),
        "age": int(row["Age"]),# age 
        "gender": row["Gender"],# other user property
        "location": row["Location"]# location
    }
    
    item_properties = {
        "name": row["Item Purchased"],
        "category": row["Category"],
        "size": row["Size"],
        "color": row["Color"],
        "season": row["Season"]
    }
    rel_properties = {
        "amount": int(row["Purchase Amount (USD)"]),
        "rating": float(row["Review Rating"]),
        "subscription": row["Subscription Status"],
        "payment_method": row["Payment Method"],
        "shipping": row["Shipping Type"],
        "discount": row["Discount Applied"],
        "promo_code": row["Promo Code Used"],
        "previous_purchases": int(row["Previous Purchases"]),
        "preferred_payment": row["Preferred Payment Method"],
        "frequency": row["Frequency of Purchases"]
    }

    conn.execute_query("""  
    MERGE (u:User {user_id: $user_id})
    SET u.age = $age, u.gender = $gender, u.location = $location
    """, user_properties) # 

    conn.execute_query("""
    MERGE (i:Item {name: $name})
    SET i.category = $category, i.size = $size, i.color = $color, i.season = $season
    """, item_properties)

    conn.execute_query("""
    MATCH (u:User {user_id: $user_id}), (i:Item {name: $name})
    MERGE (u)-[r:PURCHASED]->(i)
    SET r.amount = $amount, r.rating = $rating,
        r.subscription = $subscription, r.payment_method = $payment_method,
        r.shipping = $shipping, r.discount = $discount, r.promo_code = $promo_code,
        r.previous_purchases = $previous_purchases, r.preferred_payment = $preferred_payment,
        r.frequency = $frequency
    """, {**user_properties, **item_properties, **rel_properties}) # Reference：https://neo4j.com/docs/cypher-manual/current/syntax/parameters/

In [9]:
records = conn.execute_query("""
MATCH (u:User)-[r:PURCHASED]->(i:Item)
RETURN u.user_id, u.age, i.name, r.amount
LIMIT 5
""")
print(records)


[{'u.user_id': 1553, 'u.age': 54, 'i.name': 'Blouse', 'r.amount': 49}, {'u.user_id': 3847, 'u.age': 57, 'i.name': 'Blouse', 'r.amount': 58}, {'u.user_id': 3829, 'u.age': 42, 'i.name': 'Blouse', 'r.amount': 82}, {'u.user_id': 3808, 'u.age': 38, 'i.name': 'Blouse', 'r.amount': 39}, {'u.user_id': 3802, 'u.age': 26, 'i.name': 'Blouse', 'r.amount': 84}]


## Q2 output

Refer to q2.png

# Q3

## Q3 code

According to https://neo4j.com/docs/cypher-manual/current/subqueries/count/

In [10]:
# Reference: https://neo4j.com/docs/cypher-manual/current/subqueries/count/
def percentage_users_over_40(connection):
    query = """
    MATCH (u:User)
    WITH COUNT(u) AS total,
         COUNT(CASE WHEN u.age >= 40 THEN 1 END) AS over40
    RETURN over40 * 100.0 / total
    """
    result = connection.execute_query(query)
    return result

print(percentage_users_over_40(conn))

[{'over40 * 100.0 / total': 58.92307692307692}]


## Q3 output

Refer to q3.png

# Q4

## Q4 code

According to https://neo4j.com/docs/cypher-manual/current/subqueries/count/

According to https://neo4j.com/docs/cypher-manual/current/clauses/limit/#limit-standalone-clause

In [11]:
# Reference: Using COUNT as a grouping key-ORDER BY numDogs
def purchased_items_California(connection):
    query = """
    MATCH (u:User)-[r:PURCHASED]->(i:Item)
    WHERE u.location = "California"
    RETURN i.name AS item, count(*) AS purchases
    ORDER BY purchases DESC
    """
    result = connection.execute_query(query)
    return result
print(purchased_items_California(conn))

[{'item': 'Jeans', 'purchases': 7}, {'item': 'Dress', 'purchases': 7}, {'item': 'Shorts', 'purchases': 6}, {'item': 'Sandals', 'purchases': 5}, {'item': 'Shirt', 'purchases': 5}, {'item': 'Sunglasses', 'purchases': 5}, {'item': 'Pants', 'purchases': 5}, {'item': 'Jewelry', 'purchases': 5}, {'item': 'Backpack', 'purchases': 5}, {'item': 'Sweater', 'purchases': 4}, {'item': 'Handbag', 'purchases': 4}, {'item': 'Skirt', 'purchases': 4}, {'item': 'Scarf', 'purchases': 4}, {'item': 'Boots', 'purchases': 4}, {'item': 'Coat', 'purchases': 3}, {'item': 'Jacket', 'purchases': 3}, {'item': 'Hoodie', 'purchases': 3}, {'item': 'T-shirt', 'purchases': 3}, {'item': 'Hat', 'purchases': 3}, {'item': 'Belt', 'purchases': 3}, {'item': 'Blouse', 'purchases': 2}, {'item': 'Sneakers', 'purchases': 2}, {'item': 'Gloves', 'purchases': 2}, {'item': 'Socks', 'purchases': 1}]


In [12]:
# Reference https://neo4j.com/docs/cypher-manual/current/clauses/limit/#limit-standalone-clause
# Using LIMIT as a standalone clause
def the_most_purchased_items_California(connection):
    query = """
    MATCH (u:User)-[r:PURCHASED]->(i:Item)
    WHERE u.location = "California"
    RETURN i.name AS item, count(*) AS purchases
    ORDER BY purchases DESC 
    LIMIT 2
    """
    # Reference:https://neo4j.com/docs/cypher-manual/current/clauses/order-by/
    # Ascending and descending order
    result = connection.execute_query(query)
    return result
print(the_most_purchased_items_California(conn))

[{'item': 'Jeans', 'purchases': 7}, {'item': 'Dress', 'purchases': 7}]


## Q4 output

Refer to q4_1.png, q4_2.png

# Q5

## Q5 code

According to https://neo4j.com/docs/cypher-manual/current/functions/aggregating/#functions-count
Debug by GPT, for the first time type" RETURN r.shipping as shipping, count(*) as shipping"

In [13]:
# Reference: https://neo4j.com/docs/cypher-manual/current/functions/aggregating/#functions-count
# -count(*) includes rows returning null
def the_most_purchased_season(connection):
    query ="""
    MATCH (u:User)-[r:PURCHASED]->(i:Item)
    RETURN i.season AS season, count(*) AS purchases
    ORDER BY purchases DESC
    LIMIT 1
     """
    result = connection.execute_query(query)
    return result
print(the_most_purchased_season(conn))

[{'season': 'Spring', 'purchases': 1538}]


In [14]:
def the_most_popular_shipping_spring(connection):
    query ="""
    MATCH (u:User)-[r:PURCHASED]->(i:Item)
    WHERE i.season = "Spring"
    RETURN r.shipping AS shipping, count(*) AS count
    ORDER BY count DESC
    LIMIT 1
     """
    result = connection.execute_query(query)
    return result
print(the_most_popular_shipping_spring(conn))

[{'shipping': 'Standard', 'count': 272}]


## Q5 output

Refer to q5_1.png, q_5.png

# Q6

## Q6 code files

Refer to q6_1_producer.py and q6_2_consumer.py

## Q6 Messages tab

Refer to q6_1.png

## Q6 screenshot of the output of the consumer

Refer to q6_2.png

# Reference

-   lecture_3_SQL_and_SparkSQL notes for 14763 systems and Tool Chains for AI Engineers
-   lecture_5_Neo4J notes for 14763 systems and Tool Chains for AI Engineers
-   https://neo4j.com/docs/cypher-manual/current/constraints/managing-constraints/
-   https://neo4j.com/docs/python-manual/current/#python-driver-session-run
-   https://neo4j.com/docs/cypher-manual/current/subqueries/count/
-   https://neo4j.com/docs/cypher-manual/current/clauses/limit/#limit-standalone-clause
-   https://neo4j.com/docs/cypher-manual/current/functions/aggregating/#functions-count
-   https://developers.google.com/youtube/v3/docs/commentThreads/list#properties
-   https://medium.com/@rodolfo.antonio.sep/extracting-youtube-comments-with-python-a-detailed-guide-105363507a93
-   Debug by GPT, for the first time type" RETURN r.shipping as shipping, count(*) as shipping"